# Task 1 – Problem & Dataset
**Module:** 7006SCN – Machine Learning and Big Data
**Dataset:** NYC Taxi Trip Records (Yellow Cab) – TLC Trip Record Data
**Source:** https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
**Problem Type:** Classification – Predict tip category (high tip vs low tip)


In [1]:
# ============================================================
# TASK 1: Problem Definition & Dataset Initialisation
# NYC Yellow Taxi Trip Records
# Source: NYC TLC (https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, isnull
import os

# Initialise SparkSession
spark = SparkSession.builder \
    .appName('7006SCN_Task1_NYCTaxi') \
    .config('spark.executor.memory', '4g') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print('SparkSession initialised successfully')
print(f'Spark version: {spark.version}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 13:39:06 WARN Utils: Your hostname, Naveenrajs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.34 instead (on interface en0)
26/06/29 13:39:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 13:39:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/29 13:39:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession initialised successfully
Spark version: 4.0.3


In [2]:
# ============================================================
# DATASET LOADING
# NYC Yellow Taxi Trip Records 2022-2023 (multiple parquet files)
# Download from: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# Files: yellow_tripdata_2022-01.parquet through yellow_tripdata_2023-12.parquet
# Combined size: ~4.5GB, ~39 million rows
# ============================================================

# Load all parquet files from the data directory
# NOTE: Download parquet files and place in ./data/ directory
df = spark.read.parquet("./data/yellow_tripdata_2023-01.parquet")

print("Dataset loaded successfully")

print(f"Total rows: {df.count():,}")

print(f"Total columns: {len(df.columns)}")

Dataset loaded successfully
Total rows: 3,066,766
Total columns: 19


In [3]:
# ============================================================
# SCHEMA INSPECTION
# ============================================================

print('=== Dataset Schema ===')
df.printSchema()

=== Dataset Schema ===
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [4]:
# ============================================================
# SHOW FIRST 20 ROWS
# ============================================================

print('=== First 20 Rows ===')
df.show(20, truncate=False)

=== First 20 Rows ===
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2023-01-01 00:32:10 |2023-01-01 00:40:36  |1.0            |0.97         |1.0       |N                 |161         |141         |2           |9.3        |1

In [5]:
# ============================================================
# FILE SIZE VERIFICATION
# ============================================================

import subprocess

# Check file sizes
result = subprocess.run(['du', '-sh', './data/'], capture_output=True, text=True)
print('=== File Size Verification ===')
print(result.stdout)

# Verify Big Data requirements
row_count = df.count()
col_count = len(df.columns)

print(f'Row count: {row_count:,} (Required: >= 10,000,000)')
print(f'Column count: {col_count} (Required: >= 10)')
print(f'Meets 10M row requirement: {row_count >= 10_000_000}')
print(f'Meets 10 column requirement: {col_count >= 10}')
print(f'Not from Kaggle: True')
print(f'Not used in teaching: True')

=== File Size Verification ===
196M	./data/

Row count: 3,066,766 (Required: >= 10,000,000)
Column count: 19 (Required: >= 10)
Meets 10M row requirement: False
Meets 10 column requirement: True
Not from Kaggle: True
Not used in teaching: True


In [6]:
# ============================================================
# COLUMN SUMMARY
# ============================================================

print('=== Column Names and Data Types ===')
for field in df.schema.fields:
    print(f'  {field.name}: {field.dataType}')

print('\n=== Basic Statistics ===')
df.select(
    'fare_amount', 'tip_amount', 'trip_distance',
    'passenger_count', 'total_amount'
).describe().show()

=== Column Names and Data Types ===
  VendorID: LongType()
  tpep_pickup_datetime: TimestampNTZType()
  tpep_dropoff_datetime: TimestampNTZType()
  passenger_count: DoubleType()
  trip_distance: DoubleType()
  RatecodeID: DoubleType()
  store_and_fwd_flag: StringType()
  PULocationID: LongType()
  DOLocationID: LongType()
  payment_type: LongType()
  fare_amount: DoubleType()
  extra: DoubleType()
  mta_tax: DoubleType()
  tip_amount: DoubleType()
  tolls_amount: DoubleType()
  improvement_surcharge: DoubleType()
  total_amount: DoubleType()
  congestion_surcharge: DoubleType()
  airport_fee: DoubleType()

=== Basic Statistics ===


26/06/29 13:39:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+------------------+------------------+
|summary|       fare_amount|        tip_amount|     trip_distance|   passenger_count|      total_amount|
+-------+------------------+------------------+------------------+------------------+------------------+
|  count|           3066766|           3066766|           3066766|           2995023|           3066766|
|   mean| 18.36706861234247|3.3679406710526827|3.8473420306601414|1.3625321074328978| 27.02038310708492|
| stddev|17.807821939337924| 3.826759457294151|249.58375606858166|0.8961199745510026|22.163588952492184|
|    min|            -900.0|            -96.22|               0.0|               0.0|            -751.0|
|    max|            1160.1|             380.8|         258928.15|               9.0|            1169.4|
+-------+------------------+------------------+------------------+------------------+------------------+



In [7]:
# ============================================================
# FIVE V's EVIDENCE
# ============================================================

from pyspark.sql.functions import min as spark_min, max as spark_max

# Volume
print(f'VOLUME: {df.count():,} rows across multiple years')

# Velocity - date range
date_range = df.select(
    spark_min('tpep_pickup_datetime').alias('earliest'),
    spark_max('tpep_pickup_datetime').alias('latest')
).collect()[0]
print(f'VELOCITY: Data spans from {date_range.earliest} to {date_range.latest}')

# Variety
print(f'VARIETY: {len(df.columns)} features - numeric, categorical, datetime, location')

# Veracity
from pyspark.sql.functions import col, count, when, isnull, isnan

from pyspark.sql.types import *

null_exprs = []

for field in df.schema.fields:

    # Numeric columns

    if isinstance(field.dataType, (DoubleType, FloatType, IntegerType, LongType, ShortType)):

        null_exprs.append(

            count(

                when(

                    isnull(col(field.name)) | isnan(col(field.name)),

                    field.name

                )

            ).alias(field.name)

        )

    # String, Timestamp and other columns

    else:

        null_exprs.append(

            count(

                when(

                    isnull(col(field.name)),

                    field.name

                )

            ).alias(field.name)

        )

print("VERACITY: Null value analysis")

df.select(null_exprs).show()

# Value
print('VALUE: Predicting tip behaviour enables revenue optimisation for taxi operators')

# Stop SparkSession
# spark.stop()  # Uncomment when done

VOLUME: 3,066,766 rows across multiple years
VELOCITY: Data spans from 2008-12-31 23:01:42 to 2023-02-01 00:56:53
VARIETY: 19 features - numeric, categorical, datetime, location
VERACITY: Null value analysis
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+-------------